In [18]:
import re
import torch
from difflib import SequenceMatcher
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftConfig
from trl import GRPOConfig, GRPOTrainer

In [19]:
data = [
    {
        "prompt": "Explain reinforcement learning in simple terms.",
        "reference_answer": "Reinforcement learning is a type of machine learning where an agent learns by taking actions, receiving rewards or penalties, and improving over time.",
    },
    {
        "prompt": "Write a polite email for delayed delivery.",
        "reference_answer": "We apologize for the delay in your delivery. Your order is on the way, and we appreciate your patience and understanding.",
    },
    {
        "prompt": "What is a neural network?",
        "reference_answer": "A neural network is a machine learning model inspired by the human brain. It learns patterns from data using layers of connected nodes.",
    },
    {
        "prompt": "Explain overfitting in ML.",
        "reference_answer": "Overfitting happens when a model learns the training data too closely, including noise, and performs poorly on new unseen data.",
    },
    {
        "prompt": "Explain supervised learning in simple terms.",
        "reference_answer": "Supervised learning is a machine learning method where a model learns from labeled examples and then predicts outputs for new data.",
    },
    {
        "prompt": "What is gradient descent?",
        "reference_answer": "Gradient descent is an optimization method that helps a model reduce its error by slowly adjusting its parameters in the right direction.",
    },
    {
        "prompt": "Explain classification in machine learning.",
        "reference_answer": "Classification is a machine learning task where the model assigns input data to predefined categories or classes.",
    },
    {
        "prompt": "Write a simple apology message.",
        "reference_answer": "I am sorry for the inconvenience. Thank you for your patience and understanding.",
    },
]

In [20]:
def format_prompt(example):
    return {
        "prompt": f"Question: {example["prompt"]}\nAnswer in simple English:"
    }

In [21]:
dataset = Dataset.from_list(data)
dataset = dataset.map(format_prompt)

Map: 100%|██████████| 8/8 [00:00<00:00, 4831.45 examples/s]


In [22]:
dataset

Dataset({
    features: ['prompt', 'reference_answer'],
    num_rows: 8
})

In [23]:
model_id = "HuggingFaceTB/SmolLM-135M-Instruct"

In [29]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left" # pad on the left side

In [30]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model.config.pad_token_id = tokenizer.pad_token_id # manually passing padding info to model

In [31]:
lora_config = LoraConfig(
    task_type = "CAUSAL_LM",
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

In [32]:
# Text Cleaning
def normalize_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

In [33]:
#This function safely extracts the answer text from either a normal string or chat-message format, and converts unknown formats into a string.
def get_completion_text(completion):
    """
    Works for normal string completions.
    Also safe if completion comes in chat/message format.
    """
    if isinstance(completion, str):
        return completion

    if isinstance(completion, list):
        try:
            return completion[-1]["content"]
        except Exception:
            return str(completion)

    return str(completion)

In [34]:
#It cleans both texts, compares their character/word sequence using SequenceMatcher, and returns a similarity score from 0 to 1—1 means identical and 0 means completely different.
def similarity_score(a: str, b: str) -> float:
    return SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()

In [35]:
# This function extracts unique words of at least 3 letters from both texts and returns the fraction of reference-text words (b) that also appear in the generated text (a).

# Example:

# a = "Machine learning uses data"
# b = "Machine learning learns from data"

# Common words: machine, learning, data

# So the score is:

# 3 common words ÷ 5 reference words = 0.6

def keyword_overlap_score(a: str, b: str) -> float:
    a_words = set(re.findall(r"[a-zA-Z]{3,}", normalize_text(a)))
    b_words = set(re.findall(r"[a-zA-Z]{3,}", normalize_text(b)))

    if not b_words:
        return 0.0

    return len(a_words & b_words) / len(b_words)

In [36]:
# It acts like a teacher that compares the model’s answer with the answer key and gives marks out of 5.
def correctness_reward(prompts, completions, reference_answer=None, **kwargs):
    """
    Rewards answer similarity with reference answer.
    This is okay for demo, but not ideal for real production RL.
    """
    rewards = []

    if reference_answer is None:
        return [0.0 for _ in completions]

    for completion, ref in zip(completions, reference_answer):
        text = get_completion_text(completion)

        sim = similarity_score(text, ref)
        overlap = keyword_overlap_score(text, ref)

        # Combined correctness reward: 0 to 5
        score = (2.5 * sim) + (2.5 * overlap)
        rewards.append(float(score))

    return rewards

In [37]:
# It acts like a teacher who rewards complete and properly written answers and penalizes short, vague, or unhelpful answers.
def helpfulness_reward(prompts, completions, **kwargs):
    """
    Rewards answers that look useful and complete.
    Penalizes vague/unhelpful responses.
    """
    rewards = []

    bad_phrases = [
        "i don't know",
        "no idea",
        "maybe",
        "not sure",
        "random magic",
    ]

    for completion in completions:
        text = get_completion_text(completion)
        text_norm = normalize_text(text)

        score = 0.0

        # Good answer length
        if 40 <= len(text) <= 260:
            score += 1.0
        elif len(text) < 20:
            score -= 1.0
        elif len(text) > 350:
            score -= 0.5

        # Sentence-like response
        if "." in text or "," in text:
            score += 0.5

        # Penalize vague responses
        if any(bp in text_norm for bp in bad_phrases):
            score -= 1.5

        rewards.append(float(score))

    return rewards

In [38]:
# It acts like a teacher who rewards readable, properly sized answers and penalizes unclear or repetitive output.
def clarity_reward(prompts, completions, **kwargs):
    """
    Rewards readable English-style answers.
    """
    rewards = []

    for completion in completions:
        text = get_completion_text(completion)
        text_norm = normalize_text(text)

        score = 0.0

        # Has alphabetic content
        if re.search(r"[A-Za-z]", text):
            score += 0.5

        # Not too short / not too long
        if 30 <= len(text) <= 280:
            score += 0.5
        else:
            score -= 0.5

        # Penalize repeated junk
        if re.search(r"(.)\1{8,}", text_norm):
            score -= 1.0

        rewards.append(float(score))

    return rewards

In [39]:
training_args = GRPOConfig(
    output_dir="grpo_output",

    learning_rate=1e-5,

    # Important:
    # In TRL GRPO, per_device_train_batch_size should be divisible by num_generations.
    # So we keep both as 4 for simple single-GPU Colab demo.
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    num_generations=4,

    max_completion_length=80,

    temperature=0.9,
    top_p=0.95,

    # KL penalty. 0.0 is default in recent TRL GRPO, but 0.01 is okay for teaching demo.
    beta=0.01,

    max_steps=10,

    logging_steps=1,
    save_steps=10,

    remove_unused_columns=False,
    report_to="none",

    gradient_checkpointing=False,

    fp16=torch.cuda.is_available(),
)

In [40]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        correctness_reward,
        helpfulness_reward,
        clarity_reward,
    ],
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,
)

In [41]:
import trl
import inspect
from trl import GRPOConfig

print("TRL version:", trl.__version__)
print(inspect.signature(GRPOConfig.__init__))

TRL version: 0.22.2
(self, output_dir=None, overwrite_output_dir=None, do_train=False, do_eval=False, do_predict=False, eval_strategy='no', prediction_loss_only=False, per_device_train_batch_size=4, per_device_eval_batch_size=4, per_gpu_train_batch_size=None, per_gpu_eval_batch_size=None, gradient_accumulation_steps=2, eval_accumulation_steps=2, eval_delay=0, torch_empty_cache_steps=250, learning_rate=5e-05, weight_decay=0.001, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, max_grad_norm=1.0, num_train_epochs=3.0, max_steps=-1, lr_scheduler_type='linear', warmup_ratio=0.1, warmup_steps=0, log_level='passive', log_level_replica='warning', log_on_each_node=True, logging_dir=None, logging_strategy='steps', logging_first_step=False, logging_steps=1, logging_nan_inf_filter=False, save_strategy='steps', save_steps=500, save_total_limit=None, save_safetensors=True, save_on_each_node=False, save_only_model=False, restore_callback_states_from_checkpoint=False, no_cuda=False, use_cpu=Fals

In [42]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8 | Num Epochs = 2 | Total steps = 10
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 134,515,008 of 134,515,008 (100.00% trained)


AttributeError: 'LlamaForCausalLM' object has no attribute 'for_training'

In [43]:
trainer.save_model("grpo_output_final")
tokenizer.save_pretrained("grpo_output_final")

('grpo_output_final/tokenizer_config.json',
 'grpo_output_final/special_tokens_map.json',
 'grpo_output_final/chat_template.jinja',
 'grpo_output_final/vocab.json',
 'grpo_output_final/merges.txt',
 'grpo_output_final/added_tokens.json',
 'grpo_output_final/tokenizer.json')

In [44]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
lora_rank = 16

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.6,
)

ImportError: Unsloth: Please install vLLM before enabling `fast_inference`!
You can do this in a terminal via `pip install vllm`